# Fase 3 — Semana 2: pipeline de preprocesamiento en clases

**Grupo 4 · MCDI500 · Encuesta Nacional de Salud 2016-2017**

En la Sumativa 1 dejamos listo un conjunto de 5.511 personas para
estudiar cómo se asocian edad, sexo, escolaridad, ingreso y zona con
cinco indicadores de riesgo cardiovascular: hipertensión, diabetes,
colesterol alto, índice de masa corporal y actividad física. Ese
preprocesamiento vivía en funciones sueltas dentro de un notebook.

En esta entrega reescribimos esos mismos pasos como clases que
comparten una interfaz común. El criterio de éxito es concreto: el
pipeline con clases debe entregar exactamente el mismo conjunto que
guardamos en la Fase 2.

| Sección | Contenido |
|---|---|
| 1 | Configuración y carga del conjunto elegible F1-F2 |
| 2 | Pipeline de preprocesamiento en clases |
| 3 | Verificación contra el resultado de la Fase 2 |
| 4 | Validación: caso normal, casos límite y excepciones |
| 5 | Eficiencia: tiempo y memoria |
| 6 | Patrón de diseño Strategy aplicado a la imputación |
| 7 | Arquitectura y conclusiones |

## 1. Configuración y carga del conjunto elegible F1-F2

Partimos del archivo filtrado por ponderador en la Fase 2, antes de
la limpieza. Así las clases tienen que reproducir todo el
preprocesamiento, y el resultado se puede comparar con el conjunto
final guardado en esa fase. Las columnas se agrupan según su rol en
el estudio: predictoras sociodemográficas, indicadores de riesgo y
variables del diseño muestral.

In [ ]:
RUTA_DATOS = "data/processed/ens_variables_f1f2.xlsx"
COLUMNA_ID = "IdEncuesta"

# Predictoras sociodemográficas
COLUMNAS_PREDICTORAS_CONTINUAS = ["Edad", "anos_estudio_MINSAL_1", "as27"]
COLUMNAS_PREDICTORAS_NOMINALES = ["Sexo", "Zona"]
COLUMNAS_PREDICTORAS_ORDINALES = ["as28"]

# Indicadores de riesgo cardiovascular (se analizan por separado)
COLUMNAS_RESULTADO_BINARIAS = ["HTA"]
COLUMNAS_RESULTADO_NOMINALES = ["di3", "dis2"]
COLUMNAS_RESULTADO_ORDINALES = ["GPAQ"]
COLUMNAS_RESULTADO_CONTINUAS = ["IMC"]

# Diseño muestral: se conservan sin transformar
COLUMNAS_DISENO_MUESTRAL = ["Fexp_F1F2p_Corr", "Conglomerado", "Estrato"]

SEMILLA = 2026

COLUMNAS_ESPERADAS = (
    [COLUMNA_ID]
    + COLUMNAS_PREDICTORAS_CONTINUAS + COLUMNAS_PREDICTORAS_NOMINALES
    + COLUMNAS_PREDICTORAS_ORDINALES + COLUMNAS_RESULTADO_BINARIAS
    + COLUMNAS_RESULTADO_NOMINALES + COLUMNAS_RESULTADO_ORDINALES
    + COLUMNAS_RESULTADO_CONTINUAS + COLUMNAS_DISENO_MUESTRAL
)
print("Columnas declaradas:", len(COLUMNAS_ESPERADAS))

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# La raíz se busca aquí porque es la que permite importar src/;
# por eso no puede venir desde el propio src/carga.py.
RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists()), None)
if RAIZ is None:
    raise FileNotFoundError("No se encontró la raíz del repositorio (.git).")
sys.path.append(str(RAIZ / "src"))

from carga import cargar_conjunto, perfilar
from transformador import Transformador

np.random.seed(SEMILLA)
print("pandas", pd.__version__, "· NumPy", np.__version__, "· semilla", SEMILLA)

In [ ]:
datos = cargar_conjunto(RAIZ / RUTA_DATOS, COLUMNAS_ESPERADAS)
perfil = perfilar(datos)
perfil[perfil["nulos"] > 0]

El conjunto tiene 5.520 personas. Los nulos se concentran en `as27`
(995), `GPAQ` (196), `anos_estudio_MINSAL_1` (47), `IMC` (37) y `HTA`
(9). La no respuesta de `as28` no aparece aquí porque está codificada
como -9999: el pipeline debe convertirla en nulo antes de imputar.

## 2. Pipeline de preprocesamiento en clases

Cada paso de limpieza de la Fase 2 se reescribe como una subclase de
`Transformador` (`src/transformador.py`). La clase base fija el orden
de uso: `ajustar()` calcula los parámetros con el conjunto de
referencia y `transformar()` los aplica sobre una copia, sin volver a
calcularlos. Cada subclase solo define qué calcula (`aprender()`) y
cómo lo usa (`aplicar()`). Los pasos concretos de imputación,
codificación y escalamiento se incorporan en las subsecciones
siguientes.

## 2.1 Preparar los datos antes de imputar

Antes de rellenar cualquier hueco hay que resolver dos cosas que en la
Fase 2 se hicieron a mano.

La primera son los códigos de no respuesta. En `as28` (tramo de ingreso
del hogar) hay 818 personas con el valor -9999, que pandas trata como
un ingreso más. `MarcadorNoRespuesta` los convierte en nulos, pero
antes deja anotado en `as28_no_responde` quién no respondió: no
declarar el ingreso puede tener un significado propio y no conviene
perderlo.

La segunda son las 9 personas sin diagnóstico de hipertensión (`HTA`).
Un diagnóstico no se puede estimar sin afirmar algo que nadie declaró,
así que `EliminadorFilasNulas` las quita en lugar de rellenarlas.

Las dos clases heredan de `Transformador`: solo definen qué calculan y
cómo lo aplican. El orden de uso (ajustar antes de transformar) lo
controla la clase base.

In [ ]:
from imputadores import (
    MarcadorNoRespuesta, EliminadorFilasNulas, ImputadorFlexible,
    PorMedia, PorMediana, PorModa, PorMedianaDeTramo, comparar_estrategias,
)

marcador = MarcadorNoRespuesta("as28")
marcado = marcador.ajustar_transformar(datos)
print(marcador)
print("Personas con código -9999 en as28 (antes):", int((datos["as28"] == -9999).sum()))
print("Nulos en as28 (después):", int(marcado["as28"].isna().sum()))
print("Marcadas en as28_no_responde:", int(marcado["as28_no_responde"].sum()))

eliminador = EliminadorFilasNulas("HTA")
sin_hta = eliminador.ajustar_transformar(marcado)
print(eliminador)
print(f"Filas: {len(marcado)} -> {len(sin_hta)} ({len(marcado) - len(sin_hta)} eliminadas por HTA nulo)")
print("as28_no_responde tras eliminar:", int(sin_hta["as28_no_responde"].sum()))

Se marcaron 818 personas, pero después de quitar las 9 filas sin `HTA`
quedan 816: dos de esas nueve también tenían -9999 en `as28`.

El orden importa. La bandera se crea antes de borrar los códigos, y las
filas se eliminan antes de imputar, porque en la Fase 2 las medianas se
calcularon sobre las 5.511 personas restantes. Si se imputara antes, los
valores de relleno cambiarían un poco y el resultado ya no coincidiría.

## 2.2 Imputación: una clase y varias formas de rellenar

`ImputadorFlexible` recibe la columna y una estrategia, y no sabe
rellenar por sí sola: le pide a la estrategia que calcule con qué y que
lo aplique. Las decisiones son las de la Fase 2:

| Columna | Estrategia | Por qué |
|---|---|---|
| `IMC`, `anos_estudio_MINSAL_1` | `PorMediana` | Distribuciones asimétricas |
| `GPAQ` | `PorModa` | Es ordinal (3,6 % de nulos): la moda conserva una categoría real |
| `as27` | `PorMedianaDeTramo` sobre `as28` | Solo se imputan las 177 personas que sí declararon su tramo; las 816 que no respondieron ninguno de los dos datos quedan sin valor |

In [ ]:
pasos_imputacion = [
    ImputadorFlexible("IMC", PorMediana()),
    ImputadorFlexible("anos_estudio_MINSAL_1", PorMediana()),
    ImputadorFlexible("as27", PorMedianaDeTramo("as28")),
    ImputadorFlexible("GPAQ", PorModa()),
]

imputado = sin_hta
for paso in pasos_imputacion:
    nulos_antes = int(imputado[paso.columna].isna().sum())
    imputado = paso.ajustar_transformar(imputado)   # misma llamada para todos
    nulos_despues = int(imputado[paso.columna].isna().sum())
    print(f"{paso.nombre:<40} nulos: {nulos_antes:>4} -> {nulos_despues}")

n_imputadas = int(imputado["as27_imputado"].sum())
n_sin_dato = int(imputado["as27"].isna().sum())
print("as27 imputadas por tramo:", n_imputadas)
print("as27 que quedan sin dato:", n_sin_dato)

assert n_imputadas == 177 and n_sin_dato == 816
assert imputado[["IMC", "anos_estudio_MINSAL_1", "GPAQ"]].isna().sum().sum() == 0
assert len(imputado) == len(sin_hta), "La imputación no debe cambiar el número de filas"
print("Verificado: los conteos coinciden con los de la Fase 2.")

Herencia, polimorfismo y encapsulamiento se ven así en este código:

- **Herencia:** los tres tipos de paso parten de `Transformador`; ninguno
  reescribió el manejo del estado.
- **Polimorfismo:** todos los pasos se usan con la misma llamada,
  `ajustar_transformar`, y `ImputadorFlexible` pide `calcular` y
  `rellenar` a la estrategia sin saber si es media, mediana, moda o
  por tramo.
- **Encapsulamiento:** lo aprendido queda guardado en `_parametros`; desde
  fuera solo se obtiene una copia, y usar un paso sin ajustarlo produce
  un error.

### 2.3 Comprobación del encapsulamiento

Las tres pruebas siguientes muestran el estado protegido. Primero se
transforma sin haber ajustado, algo que debe fallar (el error se captura
a propósito). Luego se usa el orden correcto, aprendiendo con una parte
de las personas y aplicando a la otra. Por último se intenta reemplazar
desde fuera lo que el paso aprendió.

In [ ]:
# 1) Transformar antes de ajustar: debe fallar, con un mensaje claro
paso_nuevo = ImputadorFlexible("IMC", PorMediana())
try:
    paso_nuevo.transformar(datos)
except RuntimeError as error:
    print("RuntimeError:", error)

# 2) Orden correcto: se aprende en entrenamiento y se aplica a prueba
entrenamiento = sin_hta.sample(frac=0.8, random_state=SEMILLA)
prueba = sin_hta.drop(index=entrenamiento.index)

paso = ImputadorFlexible("IMC", PorMediana()).ajustar(entrenamiento)
prueba_lista = paso.transformar(prueba)

print("\nMediana aprendida en entrenamiento   :", round(paso.parametros["valor"], 2))
print("Mediana propia del conjunto de prueba:", round(prueba["IMC"].median(), 2))
print("Nulos de IMC en prueba tras imputar  :", int(prueba_lista["IMC"].isna().sum()))

# 3) Lo aprendido no se puede reemplazar desde fuera
try:
    paso.parametros = {"valor": -1}
except AttributeError as error:
    print("\nAttributeError:", error)

Sin la comprobación previa, aplicar un paso antes de ajustarlo fallaría
de forma confusa o, peor, seguiría de largo sin avisar. Aquí el problema
aparece de inmediato y con una explicación.

Las dos medianas son parecidas, pero no idénticas. Lo importante es cuál
se usa: a la prueba se le aplica la del entrenamiento, así sus propios
datos no influyen en el relleno, aunque la diferencia sea pequeña.

## 6. Patrón de diseño Strategy aplicado a la imputación

Para `as27` había varias formas razonables de rellenar los nulos. Con
Strategy, cada forma es una clase con los mismos dos métodos
(`calcular` y `rellenar`), e `ImputadorFlexible` la recibe como
parámetro. Así se pueden comparar alternativas cambiando solo el
objeto que se entrega. Se comparan tres sobre `as27`: media, mediana
y mediana por tramo de `as28`.
P

In [ ]:
estrategias_as27 = [PorMedia(), PorMediana(), PorMedianaDeTramo("as28")]
comparacion = comparar_estrategias(sin_hta, "as27", estrategias_as27)
comparacion

La media y la mediana rellenan los 993 nulos y achican la dispersión
alrededor de un 9 %, porque llenan también a las 816 personas que no
declararon ingreso. La mediana por tramo casi no la altera (-0,70 %),
pero solo rellena 177: las otras 816 quedan sin valor.

La comparación no es de igual contra igual. Se elige la mediana por
tramo porque usa un dato que la persona sí entregó (su tramo) y evita
inventar un ingreso para quienes no respondieron nada. Su límite es que
esa no respuesta podría no ser aleatoria, y eso sigue como limitación
para la interpretación de los resultados.

La comparación anterior no requirió modificar `ImputadorFlexible`. La
celda siguiente lo muestra de forma directa: la misma clase se usa con
las tres estrategias y solo cambia el objeto que recibe.

In [ ]:
for estrategia in estrategias_as27:
    paso = ImputadorFlexible("as27", estrategia)
    resultado = paso.ajustar_transformar(sin_hta)
    print(f"{estrategia.etiqueta:<20} nulos restantes en as27: {int(resultado['as27'].isna().sum())}")

**Qué habría pasado sin el patrón.** En la Fase 2, cada alternativa
para `as27` significó una función distinta o una rama `if` dentro de
`imputar_nulos_numericos`. Sumar la mediana por tramo habría obligado
a modificar esa función y el código que la llama, con el riesgo de
alterar la imputación de otras columnas.

**Por qué Strategy y no otro patrón.** El problema de esta parte es
tener varias formas intercambiables de hacer lo mismo sobre una
columna. Factory, Observer y Singleton resuelven problemas distintos
(decidir qué objeto construir, registrar eventos o compartir una única
configuración) que no aparecen en esta etapa del proyecto.